# 🎬 CineMatch: Movie Recommendation System

A complete, reproducible pipeline: **data understanding → preprocessing → EDA → feature engineering → models → evaluation → deployment artifacts → project report**.

| Step | What happens |
|------|--------------|
| 1-2 | Setup, shared code modules (`features.py`, `recommender.py`) |
| 3-4 | Load and merge the TMDB data, preprocessing |
| 5 | Exploratory data analysis (figures saved for the report) |
| 6 | Bag-of-Words, TF-IDF and sentence-embedding features |
| 7 | Collaborative filtering with MovieLens |
| 8 | Offline evaluation and hybrid-weight tuning |
| 9 | Save the artifacts used by the Streamlit app |
| 10 | Generate the project report (`.docx`) from the real results |

**Before running:** put `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv` in a folder named `datasets/` next to this notebook (in Colab: upload the folder). MovieLens downloads automatically. Run the cells **top to bottom**.

## 1. Setup

In [ ]:
!pip -q install numpy pandas scipy scikit-learn nltk sentence-transformers matplotlib python-docx

In [ ]:
import os, io, json, random, zipfile, urllib.request
from itertools import product
import numpy as np, pandas as pd, scipy.sparse as sp
import matplotlib
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_DIR, ART_DIR, FIG_DIR = "datasets", "artifacts", "figures"
for d in (ART_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
BLUE = "#4F81BD"

## 2. Shared code modules
The notebook and the web app use exactly the same code, so what is evaluated here is what gets deployed.

In [ ]:
%%writefile features.py
"""Data loading and feature engineering for the TMDB 5000 dataset.

Shared by the training notebook and by the Streamlit app (which can rebuild a lightweight
model straight from the CSV files if the full trained artifacts are not present).
"""
import ast
import json
import os

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize

try:
    from nltk.stem.porter import PorterStemmer
    _stem = PorterStemmer().stem
except Exception:                      # nltk missing -> skip stemming instead of crashing
    _stem = lambda w: w

MOVIES_CSV, CREDITS_CSV = "tmdb_5000_movies.csv", "tmdb_5000_credits.csv"


def find_data_dir(candidates=("datasets", "data", ".", "..")):
    for d in candidates:
        if os.path.exists(os.path.join(d, MOVIES_CSV)) and os.path.exists(os.path.join(d, CREDITS_CSV)):
            return d
    return None


def parse(s):
    """The TMDB csv stores lists as JSON text, e.g. '[{"id": 28, "name": "Action"}]'."""
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return []


def load_tmdb(data_dir):
    """Merge movies + credits on the unique movie id and extract clean list columns."""
    movies = pd.read_csv(os.path.join(data_dir, MOVIES_CSV))
    credits = pd.read_csv(os.path.join(data_dir, CREDITS_CSV), engine="python")
    credits = credits.rename(columns={"movie_id": "id"}).drop(columns=["title"])
    df = movies.merge(credits, on="id", how="inner", validate="one_to_one")
    df["overview"] = df["overview"].fillna("")
    df = df.dropna(subset=["release_date"]).reset_index(drop=True)
    df["year"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year.astype(int)
    df["genres_l"] = df["genres"].apply(lambda s: [d["name"] for d in parse(s)])
    df["keywords_l"] = df["keywords"].apply(lambda s: [d["name"] for d in parse(s)])
    df["cast_l"] = df["cast"].apply(lambda s: [d["name"] for d in parse(s)][:3])
    df["director"] = df["crew"].apply(lambda s: next((d["name"] for d in parse(s) if d.get("job") == "Director"), ""))
    return df


def make_tags(r):
    """overview + genres + keywords + top-3 cast + director -> one stemmed lowercase string."""
    squash = lambda xs: [x.replace(" ", "") for x in xs]          # "Sam Worthington" -> "SamWorthington"
    toks = (r["overview"].split() + squash(r["genres_l"]) + squash(r["keywords_l"])
            + squash(r["cast_l"]) + squash([r["director"]] if r["director"] else []))
    return " ".join(_stem(t) for t in " ".join(toks).lower().split())


def rich_text(r):
    """Natural-language description used by the sentence-embedding model (no stemming)."""
    return (f"{r['title']}. Genres: {', '.join(r['genres_l'])}. {r['overview']} "
            f"Keywords: {', '.join(r['keywords_l'])}. Directed by {r['director']}. Starring {', '.join(r['cast_l'])}.")


def add_quality_and_labels(df):
    """IMDb-style weighted rating + log popularity -> 'quality' in 0..1, plus unique dropdown labels."""
    C = df["vote_average"].mean()
    m = df["vote_count"].quantile(0.70)
    v, R = df["vote_count"], df["vote_average"]
    df["weighted_rating"] = (v / (v + m)) * R + (m / (v + m)) * C
    mm = lambda s: (s - s.min()) / (s.max() - s.min())
    df["quality"] = 0.7 * mm(df["weighted_rating"]) + 0.3 * mm(np.log1p(df["popularity"]))
    df["label"] = df["title"] + " (" + df["year"].astype(str) + ")"
    dup = df["label"].duplicated(keep=False)
    df.loc[dup, "label"] = df.loc[dup, "label"] + " [id " + df.loc[dup, "id"].astype(str) + "]"
    return df


META_COLS = ["id", "title", "label", "year", "genres_l", "keywords_l", "cast_l", "director", "overview",
             "vote_average", "vote_count", "popularity", "original_language", "quality"]


def build_features(df):
    """Adds tags/text/quality and returns (df, meta, count_matrix, tfidf_matrix). Rows are L2-normalised."""
    df = df.copy()
    df["tags"] = df.apply(make_tags, axis=1)
    df["text"] = df.apply(rich_text, axis=1)
    df = add_quality_and_labels(df)
    X_count = sp.csr_matrix(normalize(CountVectorizer(max_features=5000, stop_words="english").fit_transform(df["tags"])))
    X_tfidf = sp.csr_matrix(normalize(TfidfVectorizer(max_features=5000, stop_words="english",
                                                      sublinear_tf=True).fit_transform(df["tags"])))
    return df, df[META_COLS].copy(), X_count, X_tfidf

In [ ]:
%%writefile recommender.py
"""Recommendation engine shared by the notebook (evaluation) and the Streamlit app.

Signals (each is optional except count/tfidf):
  count  - Bag-of-Words cosine similarity            (original baseline)
  tfidf  - TF-IDF cosine similarity
  embed  - sentence-embedding cosine similarity      (needs artifacts from the notebook)
  cf     - collaborative filtering (MovieLens SVD)   (needs artifacts from the notebook)
  quality- popularity/rating prior
  hybrid - weighted blend of the signals above
Similarities are computed on the fly for the query movie(s), so no N x N matrix is stored.
"""
import json
import os

import numpy as np
import pandas as pd
import scipy.sparse as sp

DEFAULT_WEIGHTS = {"tfidf": 0.25, "embed": 0.35, "cf": 0.30, "quality": 0.10}
LITE_WEIGHTS = {"tfidf": 0.85, "quality": 0.15}
LIST_COLS = ["genres_l", "keywords_l", "cast_l"]


def _minmax(v, mask):
    """Scale scores to 0-1 using only the candidate movies (mask == True)."""
    if not mask.any():
        return np.zeros_like(v, dtype=float)
    x = np.where(mask, v, np.nan)
    lo, hi = np.nanmin(x), np.nanmax(x)
    out = (v - lo) / (hi - lo) if hi > lo else np.zeros_like(v, dtype=float)
    return np.where(mask, out, 0.0)


def top_n(scores, n):
    """Indices of the n best scores (descending); -inf entries are dropped."""
    n = max(1, min(n, len(scores) - 1))
    idx = np.argpartition(-scores, n - 1)[:n]
    idx = idx[np.argsort(-scores[idx])]
    return idx[np.isfinite(scores[idx])]


def _as_list(q):
    return [int(x) for x in np.atleast_1d(q)]


class Recommender:
    def __init__(self, meta, count, tfidf, emb=None, cf=None, has_cf=None, weights=None, mode="full"):
        self.meta = meta.reset_index(drop=True)
        self.n = len(self.meta)
        self.count = sp.csr_matrix(count)
        self.tfidf = sp.csr_matrix(tfidf)
        self.emb = None if emb is None else np.asarray(emb, dtype=np.float32)
        self.cf = None if cf is None else np.asarray(cf, dtype=np.float32)
        self.has_cf = np.zeros(self.n, bool) if has_cf is None else np.asarray(has_cf, dtype=bool)
        self.mode = mode
        full = self.emb is not None and self.cf is not None
        self.weights = dict(weights or (DEFAULT_WEIGHTS if full else LITE_WEIGHTS))
        self.signals = ["count", "tfidf"] + (["embed"] if self.emb is not None else []) + (["cf"] if self.cf is not None else [])
        self.quality = self.meta["quality"].to_numpy(float)
        self.labels = self.meta["label"].tolist()
        self.label_to_idx = {l: i for i, l in enumerate(self.labels)}

    # ---------- save / load / lite build ----------
    def save(self, art_dir="artifacts"):
        os.makedirs(art_dir, exist_ok=True)
        m = self.meta.copy()
        for c in LIST_COLS:                      # lists -> "a|b|c" (robust across pandas versions)
            m[c] = m[c].apply(lambda xs: "|".join(xs))
        m.to_csv(f"{art_dir}/meta.csv.gz", index=False)
        sp.save_npz(f"{art_dir}/count.npz", self.count)
        sp.save_npz(f"{art_dir}/tfidf.npz", self.tfidf)
        np.save(f"{art_dir}/emb.npy", self.emb)
        np.save(f"{art_dir}/cf.npy", self.cf)
        np.save(f"{art_dir}/has_cf.npy", self.has_cf)
        with open(f"{art_dir}/weights.json", "w") as f:
            json.dump(self.weights, f, indent=2)

    @staticmethod
    def _read_meta(path):
        meta = pd.read_csv(path)
        for c in LIST_COLS:
            meta[c] = meta[c].fillna("").apply(lambda s: [x for x in s.split("|") if x])
        meta["overview"] = meta["overview"].fillna("")
        meta["director"] = meta["director"].fillna("")
        return meta

    @classmethod
    def load(cls, art_dir="artifacts"):
        weights = None
        if os.path.exists(f"{art_dir}/weights.json"):
            with open(f"{art_dir}/weights.json") as f:
                weights = json.load(f)
        return cls(cls._read_meta(f"{art_dir}/meta.csv.gz"), sp.load_npz(f"{art_dir}/count.npz"),
                   sp.load_npz(f"{art_dir}/tfidf.npz"), np.load(f"{art_dir}/emb.npy"), np.load(f"{art_dir}/cf.npy"),
                   np.load(f"{art_dir}/has_cf.npy"), weights, mode="full")

    @classmethod
    def build_lite(cls, data_dir):
        """TF-IDF + quality model built straight from the two TMDB csv files (no notebook needed)."""
        import features as ft
        df = ft.load_tmdb(data_dir)
        df, meta, X_count, X_tfidf = ft.build_features(df)
        return cls(meta, X_count, X_tfidf, mode="lite")

    # ---------- scoring ----------
    def components(self, q):
        """Raw similarity of every movie to the query movie(s) for each available signal.
        q can be one index or a list of indices (a 'taste profile': similarity to the average of the movies)."""
        idxs = _as_list(q)
        out = {}
        for name, M in (("count", self.count), ("tfidf", self.tfidf)):
            out[name] = np.asarray(M @ np.asarray(M[idxs].mean(axis=0)).ravel()).ravel()
        if self.emb is not None:
            out["embed"] = self.emb @ self.emb[idxs].mean(axis=0)
        if self.cf is not None:
            ok = [i for i in idxs if self.has_cf[i]]
            out["cf"] = self.cf @ self.cf[ok].mean(axis=0) if ok else np.zeros(self.n, dtype=np.float32)
        return out

    def filter_mask(self, genres=None, min_rating=0.0, language=None, year_range=None, pool=None):
        m = np.ones(self.n, dtype=bool) if pool is None else pool.copy()
        if genres:
            wanted = set(genres)
            m &= self.meta["genres_l"].apply(lambda g: bool(wanted.intersection(g))).to_numpy()
        if min_rating:
            m &= (self.meta["vote_average"] >= min_rating).to_numpy()
        if language and language != "Any":
            m &= (self.meta["original_language"] == language).to_numpy()
        if year_range:
            m &= self.meta["year"].between(year_range[0], year_range[1]).to_numpy()
        return m

    def score(self, q, method="hybrid", weights=None, mask=None, comps=None):
        idxs = _as_list(q)
        mask = np.ones(self.n, dtype=bool) if mask is None else mask.copy()
        mask[idxs] = False                               # never recommend the query movie(s)
        if method == "quality":
            return np.where(mask, self.quality, -np.inf)
        comps = comps or self.components(idxs)
        if method != "hybrid":
            return np.where(mask, comps[method], -np.inf)
        w = {k: v for k, v in dict(weights or self.weights).items() if k == "quality" or k in comps}
        if "cf" in w and not self.has_cf[idxs].any():    # query has no rating data -> drop the CF signal
            w["cf"] = 0.0
        total = sum(w.values())
        if total <= 0:
            return np.where(mask, self.quality, -np.inf)
        s = np.zeros(self.n)
        for k in ("tfidf", "embed", "cf"):
            if w.get(k, 0) > 0:
                v = np.clip(comps[k], 0, None) if k == "cf" else comps[k]
                s += (w[k] / total) * _minmax(v, mask)
        s += (w.get("quality", 0) / total) * self.quality
        return np.where(mask, s, -np.inf)

    def recommend(self, movie, n=10, method="hybrid", weights=None, **filters):
        """movie = label, index, or a list of them (taste profile). filters: genres, min_rating, language, year_range."""
        items = movie if isinstance(movie, (list, tuple, np.ndarray)) else [movie]
        idxs = [self.label_to_idx[m] if isinstance(m, str) else int(m) for m in items]
        comps = self.components(idxs)
        mask = self.filter_mask(**filters)
        s = self.score(idxs, method, weights, mask, comps)
        top = top_n(s, n)
        out = self.meta.iloc[top].copy()
        out["score"] = s[top]
        for k, v in comps.items():
            out[f"s_{k}"] = v[top]
        return out

    # ---------- explanations ----------
    def explain(self, q, j):
        """Human-readable reasons why movie j is similar to the query movie(s)."""
        idxs = _as_list(q)
        m = self.meta
        g, k, c = set(), set(), set()
        directors = {m.at[i, "director"] for i in idxs if m.at[i, "director"]}
        for i in idxs:
            g |= set(m.at[i, "genres_l"]); k |= set(m.at[i, "keywords_l"]); c |= set(m.at[i, "cast_l"])
        reasons = []
        sg = [x for x in m.at[j, "genres_l"] if x in g]
        if sg: reasons.append("Shared genres: " + ", ".join(sg))
        if m.at[j, "director"] and m.at[j, "director"] in directors: reasons.append(f"Same director: {m.at[j, 'director']}")
        sc = [x for x in m.at[j, "cast_l"] if x in c]
        if sc: reasons.append("Shared cast: " + ", ".join(sc))
        sk = [x for x in m.at[j, "keywords_l"] if x in k][:5]
        if sk: reasons.append("Shared themes: " + ", ".join(sk))
        return reasons

## 3. Data understanding
Two raw files: movie details and credits (cast/crew). The text columns hold lists stored as JSON strings.

In [ ]:
movies = pd.read_csv(f"{DATA_DIR}/tmdb_5000_movies.csv")
credits_raw = pd.read_csv(f"{DATA_DIR}/tmdb_5000_credits.csv", engine="python")
print("movies:", movies.shape, "| credits:", credits_raw.shape)
movies.head(3)

In [ ]:
print("Missing values per column (only columns with gaps):")
miss = movies.isna().sum(); print(miss[miss > 0])
print("\nGenres cell example :", movies.loc[0, "genres"][:90], "...")
print("Credits cell example:", credits_raw.loc[0, "cast"][:90], "...")

In [ ]:
# Why we merge on id and not on title: some titles are not unique, so a title merge silently adds rows.
n_title_merge = len(movies.merge(credits_raw.drop(columns=["movie_id"]), on="title"))
print(f"rows after merging on title: {n_title_merge}  |  movies file has: {len(movies)} rows")

## 4. Preprocessing
`features.load_tmdb` merges on the unique id, parses the JSON columns, extracts the top-3 cast and the director, and creates the release year.

In [ ]:
import features as ft
df = ft.load_tmdb(DATA_DIR)
n_missing_overview = int(movies["overview"].isna().sum())
n_missing_release = int(movies["release_date"].isna().sum())
print("after merge on id and cleaning:", df.shape)
df[["title", "year", "genres_l", "cast_l", "director"]].head(3)

In [ ]:
df, meta, X_count, X_tfidf = ft.build_features(df)
print("Tags example (stemmed, spaces removed from names):\n", df.loc[0, "tags"][:200], "...")
print("\nText for the embedding model:\n", df.loc[0, "text"][:250], "...")
print("\nBag-of-Words matrix:", X_count.shape, "| TF-IDF matrix:", X_tfidf.shape)

## 5. Exploratory data analysis
Figures are saved in `figures/` and used automatically in the report.

In [ ]:
figs = {}
def save(name):
    plt.tight_layout(); p = f"{FIG_DIR}/{name}.png"; plt.savefig(p, dpi=170); plt.show(); figs[name] = p

# 1 genres
g = df.explode("genres_l")["genres_l"].value_counts()
g.head(15)[::-1].plot(kind="barh", color=BLUE, figsize=(7, 4.5)); plt.xlabel("Number of movies"); plt.title("Movies per genre"); save("genres")
# 2 ratings
df.loc[df.vote_count >= 10, "vote_average"].plot(kind="hist", bins=np.arange(0, 10.5, 0.5), color=BLUE, edgecolor="white", figsize=(7, 4))
plt.xlabel("Average rating"); plt.ylabel("Movies"); plt.title("Distribution of ratings (10+ votes)"); save("ratings")
# 3 decades
(df.year // 10 * 10).value_counts().sort_index().plot(kind="bar", color=BLUE, figsize=(7, 4)); plt.xlabel("Decade"); plt.ylabel("Movies"); plt.title("Movies per decade"); save("decades")
# 4 languages
df["original_language"].value_counts().head(8)[::-1].plot(kind="barh", color=BLUE, figsize=(7, 4)); plt.xlabel("Movies"); plt.title("Top original languages"); save("languages")
# 5 rating vs popularity
s = df[df.vote_count >= 10]
plt.figure(figsize=(7, 4.5)); plt.scatter(np.log1p(s.popularity), s.vote_average, s=8, alpha=.4, color=BLUE)
plt.xlabel("log(1 + popularity)"); plt.ylabel("Average rating"); plt.title("Rating vs popularity"); save("popularity")
# 6 keywords
kw = Counter(k for ks in df.keywords_l for k in ks).most_common(15)
plt.figure(figsize=(7, 4.5)); plt.barh([k for k, _ in kw][::-1], [c for _, c in kw][::-1], color=BLUE); plt.xlabel("Movies"); plt.title("Most frequent keywords"); save("keywords")

stats = dict(
    n_movies_raw=len(movies), n_credits_raw=len(credits_raw), n_merged_cols=int(movies.shape[1] + credits_raw.shape[1] - 2),
    n_title_merge=n_title_merge, n_final=len(df), n_missing_overview=n_missing_overview, n_missing_release=n_missing_release,
    n_genres=int(g.size), year_min=int(df.year.min()), year_max=int(df.year.max()), avg_rating=float(df.vote_average.mean()),
    n_directors=int(df.director.replace("", np.nan).nunique()), top_genres=[(k, int(v)) for k, v in g.head(5).items()],
    pct_english=float((df.original_language == "en").mean() * 100))
stats

## 6. Embeddings (semantic features)
A pretrained sentence-transformer turns each movie description into a vector that captures its meaning. First run: about 1-3 minutes on CPU (a GPU runtime in Colab is faster).

In [ ]:
emb_path = f"{ART_DIR}/emb_cache.npy"
if os.path.exists(emb_path):
    E = np.load(emb_path)
else:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    E = model.encode(df["text"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    np.save(emb_path, E)
E = np.asarray(E, dtype=np.float32)
print("embeddings:", E.shape)

## 7. Collaborative filtering with MovieLens
TMDB has no user ratings, so the free MovieLens (small) dataset is linked to our movies through the TMDB id. Movies are described by *who liked them* using SVD on the user × movie rating matrix.

In [ ]:
ML_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
if not os.path.exists("ml-latest-small"):
    with urllib.request.urlopen(ML_URL) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
ratings = pd.read_csv("ml-latest-small/ratings.csv")
links = pd.read_csv("ml-latest-small/links.csv").dropna(subset=["tmdbId"])
links["tmdbId"] = links["tmdbId"].astype(int)
id2idx = {mid: i for i, mid in enumerate(df["id"])}
links["idx"] = links["tmdbId"].map(id2idx)
links = links.dropna(subset=["idx"])
ratings = ratings.merge(links[["movieId", "idx"]], on="movieId")
ratings["idx"] = ratings["idx"].astype(int)
print(f"{len(ratings):,} ratings on {ratings['idx'].nunique():,} of our {len(df):,} movies from {ratings['userId'].nunique()} users")

In [ ]:
SVD_K = 50
def build_cf(ratings_df, user_ids, n_items, k=SVD_K, min_ratings=5, seed=SEED):
    # L2-normalised item factors (N x k) and a mask of movies that have enough ratings.
    r = ratings_df[ratings_df["userId"].isin(user_ids)].copy()
    r["c"] = r["rating"] - r.groupby("userId")["rating"].transform("mean")        # remove each user's rating bias
    umap = {u: i for i, u in enumerate(sorted(set(user_ids)))}
    M = sp.csr_matrix((r["c"].values, (r["userId"].map(umap).values, r["idx"].values)), shape=(len(umap), n_items))
    has = np.bincount(r["idx"].values, minlength=n_items) >= min_ratings
    k = max(2, min(k, M.shape[0] - 1, M.shape[1] - 1))
    svd = TruncatedSVD(n_components=k, random_state=seed).fit(M)
    F = normalize(svd.components_.T * np.sqrt(svd.singular_values_))
    F[~has] = 0
    return F.astype(np.float32), has

# users are split so the evaluation is fair: 60% build the CF model, 20% tune the hybrid, 20% test
users = np.array(sorted(ratings["userId"].unique()))
np.random.RandomState(SEED).shuffle(users)
a, b = int(0.6 * len(users)), int(0.8 * len(users))
train_u, val_u, test_u = users[:a], users[a:b], users[b:]
cf_train, has_train = build_cf(ratings, train_u, len(df))
print("movies with enough ratings for CF:", int(has_train.sum()))

In [ ]:
from recommender import Recommender, top_n, DEFAULT_WEIGHTS
rec_eval = Recommender(meta, X_count, X_tfidf, E, cf_train, has_train)
print("evaluation recommender ready:", rec_eval.n, "movies | signals:", rec_eval.signals)

## 8. Offline evaluation and hybrid tuning
**Protocol.** For each held-out user we pick a movie they *liked* (rating ≥ 4) as the **query**, ask for 10 recommendations and check how many of the user's *other* liked movies appear.

* **Precision@10**: share of the 10 recommendations the user liked.
* **Hit-rate@10**: share of queries where at least one recommendation was liked.
* **Genre Jaccard@10**: genre overlap between query and recommendations (a content-only sanity check; it favours methods that use genres, so it is secondary).

Only movies with enough ratings are candidates, so all methods compete on the same pool. Absolute numbers look small because each user liked only a few of thousands of candidates, so compare methods with **each other**. Hybrid weights are tuned on validation users and everything is reported on unseen **test** users.

In [ ]:
pool = has_train.copy()
gsets = [set(x) for x in meta["genres_l"]]
jacc = lambda a, b: len(a & b) / len(a | b) if (a | b) else 0.0

def build_queries(user_ids, per_user=3, seed=SEED):
    rng = np.random.default_rng(seed); out = []
    for u, g_ in ratings[ratings["userId"].isin(user_ids)].groupby("userId"):
        liked = [i for i in g_.loc[g_["rating"] >= 4, "idx"].unique() if pool[i]]
        if len(liked) < 6: continue
        for q in rng.choice(liked, size=min(per_user, len(liked)), replace=False):
            out.append((int(q), set(int(x) for x in liked) - {int(q)}))
    return out

val_q, test_q = build_queries(val_u), build_queries(test_u)
val_c  = [rec_eval.components(q) for q, _ in val_q]
test_c = [rec_eval.components(q) for q, _ in test_q]
print("validation queries:", len(val_q), "| test queries:", len(test_q))

def evaluate(queries, comps_list, method, weights=None, K=10, genre=True):
    P = H = J = 0.0
    for (q, rel), comps in zip(queries, comps_list):
        top = top_n(rec_eval.score(q, method, weights, pool, comps), K)
        hits = len(rel.intersection(top.tolist()))
        P += hits / K; H += hits > 0
        if genre: J += np.mean([jacc(gsets[q], gsets[t]) for t in top])
    n = len(queries)
    return {"Precision@10": P / n, "HitRate@10": H / n, "GenreJaccard@10": J / n if genre else np.nan}

In [ ]:
def simplex_grid(keys, step=0.1):
    k = int(round(1 / step))
    for combo in product(range(k + 1), repeat=len(keys) - 1):
        if sum(combo) <= k:
            vals = list(combo) + [k - sum(combo)]
            yield {key: v / k for key, v in zip(keys, vals)}

best_w, best = None, (-1, -1)
for w in simplex_grid(["tfidf", "embed", "cf", "quality"]):
    r = evaluate(val_q, val_c, "hybrid", w, genre=False)
    if (r["Precision@10"], r["HitRate@10"]) > best:
        best, best_w = (r["Precision@10"], r["HitRate@10"]), w
print("best hybrid weights:", best_w, "| validation Precision@10 = %.4f" % best[0])

In [ ]:
names = {"quality": "Popularity only (baseline)", "count": "Bag-of-Words (original project)", "tfidf": "TF-IDF",
         "embed": "Sentence embeddings", "cf": "Collaborative filtering (SVD)", "hybrid": "Hybrid (tuned)"}
rows = []
for m_ in ["quality", "count", "tfidf", "embed", "cf", "hybrid"]:
    r = evaluate(test_q, test_c, m_, best_w if m_ == "hybrid" else None)
    rows.append({"Method": names[m_], **r})
results = pd.DataFrame(rows).set_index("Method")
results.to_csv(f"{ART_DIR}/evaluation_results.csv")
results.round(4)

In [ ]:
ax = results[["Precision@10", "HitRate@10"]].plot(kind="bar", figsize=(9, 4.5), rot=25, color=[BLUE, "#C0504D"])
ax.set_title("Recommendation quality on unseen users (higher is better)"); ax.set_ylabel("score")
save("evaluation")

## 9. Final model and app artifacts
The collaborative model is refit on **all** users (more data), then everything the app needs is saved in `artifacts/`.

In [ ]:
cf_all, has_all = build_cf(ratings, users, len(df))
final = Recommender(meta, X_count, X_tfidf, E, cf_all, has_all, weights=best_w)
final.save(ART_DIR)
if os.path.exists(f"{ART_DIR}/emb_cache.npy"): os.remove(f"{ART_DIR}/emb_cache.npy")
for f in sorted(os.listdir(ART_DIR)):
    print(f"{f:24s} {os.path.getsize(os.path.join(ART_DIR, f)) / 1e6:6.2f} MB")

In [ ]:
title = "The Dark Knight Rises"
q = int(meta.index[meta["title"] == title][0])
for m_ in ["count", "tfidf", "embed", "cf", "hybrid"]:
    print(f"{names[m_]:35s} ->", ", ".join(final.recommend(q, n=5, method=m_)["title"]))
print("\nWhy the first hybrid recommendation was chosen:")
top1 = final.recommend(q, n=1)
print(final.explain(q, int(top1.index[0])))
print("\nTaste profile (2 movies):", ", ".join(final.recommend([q, int(meta.index[meta.title == meta.title.iloc[10]][0])], n=5)["title"]))

## 10. Generate the project report
The report is written from the **real numbers and figures** produced above. It is saved as `Movie_Recommendation_Report.docx`.

In [ ]:
%%writefile make_report.py
"""Builds the project report (.docx) from the REAL results produced by the notebook."""
import os

from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_COLOR_INDEX
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

BLUE, LBLUE = RGBColor(0x36, 0x5F, 0x91), RGBColor(0x4F, 0x81, 0xBD)


def _font(style, size=None, bold=None, color=None):
    style.font.name = "Arial"
    rf = style.element.rPr.rFonts
    rf.set(qn("w:eastAsia"), "Arial")
    for a in ("w:asciiTheme", "w:hAnsiTheme", "w:eastAsiaTheme", "w:cstheme"):   # theme fonts would override Arial
        if rf.get(qn(a)) is not None:
            del rf.attrib[qn(a)]
    if size: style.font.size = Pt(size)
    if bold is not None: style.font.bold = bold
    if color is not None: style.font.color.rgb = color


def _shade(cell, fill):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto"); shd.set(qn("w:fill"), fill)
    tcPr.append(shd)


def _field(paragraph, instr, placeholder=""):
    """Insert a Word field (PAGE, TOC...) into a paragraph."""
    def fc(t):
        r = paragraph.add_run(); e = OxmlElement("w:fldChar"); e.set(qn("w:fldCharType"), t); r._r.append(e); return r
    fc("begin")
    r = paragraph.add_run(); i = OxmlElement("w:instrText"); i.set(qn("xml:space"), "preserve"); i.text = instr; r._r.append(i)
    fc("separate")
    run = paragraph.add_run(placeholder)
    fc("end")
    return run


def _fmt(x, d=4):
    return f"{x:.{d}f}"


class Report:
    def __init__(self):
        self.d = Document()
        s = self.d.styles
        _font(s["Normal"], 11)
        s["Normal"].paragraph_format.space_after = Pt(7)
        s["Normal"].paragraph_format.line_spacing = 1.25
        _font(s["Heading 1"], 15, True, BLUE)
        s["Heading 1"].paragraph_format.page_break_before = True
        s["Heading 1"].paragraph_format.space_after = Pt(10)
        _font(s["Heading 2"], 12, True, LBLUE)
        s["Heading 2"].paragraph_format.space_before = Pt(10)
        _font(s["List Bullet"], 11)
        for sec in self.d.sections:
            sec.page_width, sec.page_height = Inches(8.5), Inches(11)
            sec.left_margin = sec.right_margin = sec.top_margin = sec.bottom_margin = Inches(1)

    # --- building blocks
    def h1(self, t): return self.d.add_heading(t, 1)
    def h2(self, t): return self.d.add_heading(t, 2)
    def p(self, t, bold=False, italic=False, center=False):
        para = self.d.add_paragraph(); r = para.add_run(t); r.bold, r.italic = bold, italic
        if center: para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        return para
    def bullets(self, items):
        for t in items: self.d.add_paragraph(t, style="List Bullet").paragraph_format.space_after = Pt(3)
    def note(self, t):
        para = self.d.add_paragraph(); r = para.add_run(t); r.italic = True; r.font.highlight_color = WD_COLOR_INDEX.YELLOW
    def table(self, rows, widths=None, header=True):
        t = self.d.add_table(rows=len(rows), cols=len(rows[0])); t.style = "Table Grid"; t.alignment = WD_TABLE_ALIGNMENT.CENTER
        for i, row in enumerate(rows):
            for j, val in enumerate(row):
                c = t.cell(i, j); c.text = ""
                run = c.paragraphs[0].add_run(str(val)); run.font.size = Pt(10)
                c.paragraphs[0].paragraph_format.space_after = Pt(2)
                if header and i == 0:
                    run.bold = True; run.font.color.rgb = RGBColor(255, 255, 255); _shade(c, "4F81BD")
                elif i % 2 == 0:
                    _shade(c, "EEF3FA")
                if widths: c.width = Inches(widths[j])
        self.d.add_paragraph().paragraph_format.space_after = Pt(4)
    def figure(self, path, caption, width=5.8):
        if path and os.path.exists(path):
            self.d.add_picture(path, width=Inches(width)); self.d.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
            self.d.paragraphs[-1].paragraph_format.keep_with_next = True      # keep figure and caption together
            c = self.p(caption, italic=True, center=True); c.runs[0].font.size = Pt(9.5)
    def placeholder(self, t):
        tb = self.d.add_table(rows=1, cols=1); tb.style = "Table Grid"; c = tb.cell(0, 0); _shade(c, "F5F5F5")
        c.text = ""; para = c.paragraphs[0]; para.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r = para.add_run("\n\n" + t + "\n\n"); r.italic = True; r.font.color.rgb = RGBColor(0x7F, 0x7F, 0x7F)
        self.d.add_paragraph()


def _improve(a, b):
    """Plain-English comparison of metric a vs baseline b."""
    if b <= 0: return "could not be compared (baseline score is zero)"
    ch = (a - b) / b * 100
    return f"{abs(ch):.0f}% {'higher' if ch >= 0 else 'lower'} than"


def build_report(ctx, path):
    S, M, R = ctx["stats"], ctx["ml"], ctx["results"]
    best_w, fig = ctx["best_w"], ctx["figures"]
    rp = Report(); d = rp.d
    best = R["Precision@10"].idxmax()
    base_name, hyb = "Bag-of-Words (original project)", "Hybrid (tuned)"
    pop = "Popularity only (baseline)"

    # ---------- cover placeholder + index ----------
    c = d.add_paragraph(); c.paragraph_format.space_before = Pt(200); c.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = c.add_run("[ COVER PAGE ]"); r.bold = True; r.font.size = Pt(18); r.font.color.rgb = RGBColor(0x7F, 0x7F, 0x7F)
    c = d.add_paragraph("Replace this page with your cover page (title, name, institute, internship details, demo and GitHub links).")
    c.alignment = WD_ALIGN_PARAGRAPH.CENTER
    sec = d.add_section(WD_SECTION.NEW_PAGE)
    pg = OxmlElement("w:pgNumType"); pg.set(qn("w:start"), "1"); sec._sectPr.append(pg)
    sec.footer.is_linked_to_previous = False
    fp = sec.footer.paragraphs[0]; fp.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = _field(fp, "PAGE", "1"); run.bold = True; run.font.size = Pt(13); run.font.color.rgb = BLUE

    t = d.add_paragraph(); r = t.add_run("INDEX"); r.bold = True; r.font.size = Pt(20)
    toc = d.add_paragraph()
    _field(toc, 'TOC \\o "1-1" \\h \\z \\u', "Right-click here and choose “Update Field” (or press F9) to build the index.")
    upd = OxmlElement("w:updateFields"); upd.set(qn("w:val"), "true")      # ask Word to build the index on open
    anchor = next((e for e in d.settings.element if e.tag in (qn("w:hdrShapeDefaults"), qn("w:footnotePr"),
                                                              qn("w:endnotePr"), qn("w:compat"))), None)
    if anchor is not None: anchor.addprevious(upd)
    else: d.settings.element.append(upd)

    # ---------- abstract ----------
    rp.h1("Abstract")
    rp.p("Streaming platforms and online stores now offer thousands of movies, which makes choosing what to watch difficult. "
         "Recommendation systems solve this by analysing the content of movies and the behaviour of viewers to suggest titles "
         "that a person is likely to enjoy.")
    rp.p(f"This project develops a multi-signal movie recommendation system built on the TMDB 5000 Movie dataset "
         f"({S['n_final']:,} movies) and the MovieLens ratings dataset ({M['n_ratings']:,} ratings from {M['n_users']} viewers). "
         "The movies are described using their overview, genres, keywords, leading cast and director. Four families of "
         "signals are implemented and compared: Bag-of-Words similarity (the baseline), TF-IDF similarity, semantic similarity "
         "using sentence embeddings (all-MiniLM-L6-v2), and collaborative filtering based on matrix factorisation (SVD) of viewer ratings. "
         "A hybrid ranker combines these signals with a rating-and-popularity prior, and its weights are tuned on validation users.")
    rp.p(f"The methods were evaluated offline on {M['n_test_q']} queries from {M['n_test_users']} users who were not used to build or tune the models, "
         f"using Precision@10 and Hit-rate@10. The best-performing method was {best} "
         f"(Precision@10 = {_fmt(R.loc[best, 'Precision@10'])}, Hit-rate@10 = {_fmt(R.loc[best, 'HitRate@10'])}). "
         f"For comparison, the original Bag-of-Words approach scored {_fmt(R.loc[base_name, 'Precision@10'])} Precision@10.")
    rp.p("The system is deployed as an interactive Streamlit web application with recommendation by a single movie, "
         "recommendation from a taste profile of several movies, filters (genre, rating, language and release year), "
         "explanations for each recommendation, a data-exploration dashboard and a model-comparison page.")

    # ---------- chapter 1 ----------
    rp.h1("CHAPTER 1: INTRODUCTION")
    rp.h2("1.1 Introduction")
    rp.p("Machine Learning (ML) enables computers to learn patterns from data. Recommendation systems are one of its most "
         "visible applications and are used by platforms such as Netflix, YouTube and Spotify to personalise content. "
         "Two main approaches exist: content-based filtering, which recommends items similar to those a user already likes, "
         "and collaborative filtering, which recommends items liked by people with similar tastes. Modern systems combine both.")
    rp.h2("1.2 Problem Statement")
    rp.p("With thousands of movies available, users spend a long time searching for something to watch, and simple lists such as "
         "“most popular” ignore personal taste. The objective of this project is to build and evaluate a recommender that "
         "suggests relevant movies from one or several movies the user likes, and to deploy it as a usable web application.")
    rp.h2("1.3 Objectives")
    rp.bullets(["Clean, merge and explore the TMDB movie data and engineer a rich text description for every movie.",
                "Implement content-based recommenders using Bag-of-Words, TF-IDF and sentence embeddings.",
                "Implement collaborative filtering using MovieLens ratings and matrix factorisation (SVD).",
                "Combine the signals into a hybrid ranker and tune its weights on held-out validation users.",
                "Evaluate all methods quantitatively on unseen users using Precision@10 and Hit-rate@10.",
                "Deploy the best system as a Streamlit web application with filters, explanations and taste profiles."])
    rp.h2("1.4 Scope of the Project")
    rp.p("The project covers the complete machine learning workflow: data understanding, preprocessing, exploratory analysis, "
         "feature engineering, model development, offline evaluation and web deployment. Recommendations are made from the "
         f"{S['n_final']:,} movies of the TMDB 5000 dataset; collaborative filtering is available for the {M['n_cf_movies']:,} of those "
         "movies that have enough ratings in MovieLens.")
    rp.h2("1.5 Project Workflow")
    rp.figure(fig.get("workflow"), "Figure 1.1: Project workflow", 5.6)
    rp.h2("1.6 Work Plan")
    rp.p("The work was organised into the following phases:")
    rp.table([["Phase", "Activities", "Duration"],
              ["1. Study and data collection", "Literature study of recommender systems; collecting the TMDB and MovieLens datasets", "Weeks 1-2"],
              ["2. Preprocessing and EDA", "Merging, cleaning, parsing nested fields, feature engineering, exploratory analysis", "Weeks 2-3"],
              ["3. Content-based models", "Bag-of-Words, TF-IDF and sentence-embedding recommenders", "Weeks 3-5"],
              ["4. Collaborative filtering and hybrid", "MovieLens integration, SVD, hybrid ranking and weight tuning", "Weeks 5-6"],
              ["5. Evaluation", "Offline protocol, Precision@10 / Hit-rate@10, comparison of methods", "Weeks 6-7"],
              ["6. Deployment and documentation", "Streamlit application, GitHub deployment, report writing", "Weeks 7-8"]],
             widths=[1.9, 3.5, 1.1])
    rp.note("Edit the Duration column so it matches your real schedule before you submit.")

    # ---------- chapter 2 ----------
    rp.h1("CHAPTER 2: DATASET DESCRIPTION")
    rp.h2("2.1 TMDB 5000 Movie Dataset")
    rp.p("The primary dataset consists of two CSV files: tmdb_5000_movies.csv (movie details) and tmdb_5000_credits.csv (cast and crew). "
         "The files were merged on the unique movie id.")
    rp.table([["Item", "Value"],
              ["Rows in movies file / credits file", f"{S['n_movies_raw']:,} / {S['n_credits_raw']:,}"],
              ["Columns after merging", S["n_merged_cols"]],
              ["Movies used after cleaning", f"{S['n_final']:,}"],
              ["Release years covered", f"{S['year_min']} - {S['year_max']}"],
              ["Distinct genres", S["n_genres"]],
              ["Distinct directors", f"{S['n_directors']:,}"],
              ["Average rating (TMDB)", f"{S['avg_rating']:.2f} / 10"]], widths=[3.2, 3.2])
    rp.h2("2.2 MovieLens Ratings")
    rp.p("To learn from viewer behaviour, the MovieLens (small) dataset from GroupLens was used. It contains explicit star ratings "
         "from viewers, and each MovieLens movie carries a TMDB id, which allowed it to be linked to the TMDB movies.")
    rp.table([["Item", "Value"],
              ["Ratings linked to our movies", f"{M['n_ratings']:,}"],
              ["Users", M["n_users"]],
              ["TMDB movies with at least one rating", f"{M['n_movies_overlap']:,}"],
              ["Movies with enough ratings for collaborative filtering", f"{M['n_cf_movies']:,}"]], widths=[3.8, 2.6])
    rp.h2("2.3 Features Used")
    rp.bullets(["Text features: overview, genres, keywords, top three cast members, director.",
                "Numeric features: vote average, vote count, popularity, release year.",
                "Viewer features: user-movie ratings from MovieLens."])

    # ---------- chapter 3 ----------
    rp.h1("CHAPTER 3: DATA PREPROCESSING")
    rp.p("The following preprocessing steps were performed:")
    rp.bullets([f"Merged the movies and credits files on the movie id and verified a one-to-one match. Merging on the title instead produced "
                f"{S['n_title_merge']:,} rows instead of {S['n_movies_raw']:,}, because some titles are not unique, so the id was used.",
                f"Filled the {S['n_missing_overview']} missing overviews with empty text and dropped {S['n_missing_release']} movie(s) without a release date.",
                "Parsed the JSON-style text columns (genres, keywords, cast, crew) into Python lists.",
                "Kept the top three cast members and extracted the director from the crew list.",
                "Removed spaces inside names so that each person or genre becomes a single token (for example “Sam Worthington” became “SamWorthington”).",
                "Combined overview, genres, keywords, cast and director into one text field (tags), lower-cased it and applied Porter stemming.",
                "Built a separate natural-language description of each movie for the sentence-embedding model.",
                "Created a quality score by combining an IMDb-style weighted rating with log popularity.",
                "Created unique display labels such as “Title (Year)” for the dropdown menus."])
    rp.table([["Field", "Before", "After"], ["Genre", "Science Fiction", "ScienceFiction"],
              ["Cast", "Sam Worthington", "SamWorthington"], ["Director", "James Cameron", "JamesCameron"]], widths=[1.8, 2.3, 2.3])
    rp.p("The weighted rating prevents a movie with only a few votes from outranking a movie with thousands of votes:")
    rp.p("WR = (v / (v + m)) × R + (m / (v + m)) × C", italic=True, center=True)
    rp.p("where R is the movie's average rating, v its number of votes, m the 70th percentile of vote counts, and C the mean rating of all movies.")

    # ---------- chapter 4 ----------
    rp.h1("CHAPTER 4: EXPLORATORY DATA ANALYSIS")
    rp.p("Exploratory Data Analysis (EDA) was carried out to understand the dataset before modelling.")
    rp.figure(fig.get("genres"), "Figure 4.1: Number of movies per genre")
    tg = ", ".join(f"{g} ({c:,})" for g, c in S["top_genres"][:3])
    rp.p(f"The most common genres are {tg}. Most movies carry two or three genre labels, so genres are a useful but coarse signal.")
    rp.figure(fig.get("ratings"), "Figure 4.2: Distribution of ratings (movies with at least 10 votes)")
    rp.p(f"The average rating is {S['avg_rating']:.2f} out of 10, and ratings are concentrated in the middle of the scale rather than at the extremes.")
    rp.figure(fig.get("decades"), "Figure 4.3: Movies released per decade")
    rp.figure(fig.get("languages"), "Figure 4.4: Most common original languages")
    rp.p(f"{S['pct_english']:.0f}% of the movies are originally in English, so the catalog is dominated by English-language films.")
    rp.figure(fig.get("popularity"), "Figure 4.5: Rating versus popularity (log scale)")
    rp.p("Popularity and rating are only loosely related, which is why the hybrid model uses a combined quality score instead of either alone.")
    rp.figure(fig.get("keywords"), "Figure 4.6: Most frequent keywords")

    # ---------- chapter 5 ----------
    rp.h1("CHAPTER 5: MODEL DEVELOPMENT")
    rp.p("Five recommendation approaches were developed. All of them compare movies using cosine similarity, which is the cosine of the angle "
         "between two vectors; a higher value means more similar movies.")
    rp.h2("5.1 Bag-of-Words (baseline)")
    rp.p("Each movie's tags are converted into a vector of word counts using the 5,000 most frequent words with English stop words removed. This is the approach of the original project and is used as the baseline.")
    rp.h2("5.2 TF-IDF")
    rp.p("TF-IDF weights each word by how often it appears in a movie and how rare it is across all movies, so common words matter less and distinctive words such as a director's name matter more. Sub-linear term frequency was used.")
    rp.h2("5.3 Sentence Embeddings")
    rp.p(f"The pretrained model all-MiniLM-L6-v2 converts each movie's description into a {ctx['emb_dim']}-dimensional vector that captures meaning, so movies with similar stories are close even if they share few words.")
    rp.h2("5.4 Collaborative Filtering")
    rp.p(f"The user-movie rating matrix (each user's ratings centred by their average) is factorised using truncated SVD with {ctx['svd_k']} components. "
         "Movies are then compared in the latent space, so two movies are similar if the same viewers tend to like them. Users were split so that the factorisation only uses training users.")
    rp.h2("5.5 Hybrid Ranking")
    rp.p("The hybrid score is a weighted sum of the min-max scaled TF-IDF, embedding and collaborative similarities and the quality score. "
         "The weights were selected by grid search (steps of 0.1, summing to 1) on validation users.")
    rp.table([["Signal", "Tuned weight"]] + [[k, f"{v:.1f}"] for k, v in best_w.items()], widths=[3.2, 2.0])

    # ---------- chapter 6 ----------
    rp.h1("CHAPTER 6: MODEL EVALUATION")
    rp.h2("6.1 Evaluation Protocol")
    rp.p(f"Users were split into three groups: {M['n_train_users']} users to build the collaborative model, {M['n_val_users']} to tune the hybrid weights and "
         f"{M['n_test_users']} for the final test. For each test user, a movie the user liked (rating of at least 4) was used as the query, the system "
         "returned 10 recommendations, and these were compared with the user's other liked movies. Only movies with enough ratings were candidates so that all methods compete on the same pool.")
    rp.bullets(["Precision@10: the share of the 10 recommendations that the user liked.",
                "Hit-rate@10: the share of queries where at least one recommendation was liked.",
                "Genre Jaccard@10: genre overlap between the query and the recommendations (content-only sanity check)."])
    rp.h2("6.2 Results")
    rows = [["Method", "Precision@10", "Hit-rate@10", "Genre Jaccard@10"]]
    for name, r in R.iterrows():
        rows.append([name, _fmt(r["Precision@10"]), _fmt(r["HitRate@10"]), _fmt(r["GenreJaccard@10"])])
    rp.table(rows, widths=[2.7, 1.2, 1.2, 1.4])
    rp.figure(fig.get("evaluation"), "Figure 6.1: Comparison of methods on unseen users (higher is better)")
    rp.h2("6.3 Discussion")
    rp.p(f"The best method on Precision@10 was {best}. The tuned hybrid model scored {_fmt(R.loc[hyb, 'Precision@10'])}, which is "
         f"{_improve(R.loc[hyb, 'Precision@10'], R.loc[base_name, 'Precision@10'])} the original Bag-of-Words baseline ({_fmt(R.loc[base_name, 'Precision@10'])}) "
         f"and {_improve(R.loc[hyb, 'Precision@10'], R.loc[pop, 'Precision@10'])} the popularity-only baseline ({_fmt(R.loc[pop, 'Precision@10'])}).")
    rp.p("Absolute precision values are small because each user liked only a few movies out of thousands of candidates, so the methods should be compared with each other rather than with 1.0. "
         "Genre Jaccard is a secondary measure and favours methods that use genre information directly.")
    rp.h2("6.4 Sample Output")
    rp.p(f"Recommendations for {ctx['demo_title']} using the hybrid model:")
    rp.table([["Rank", "Recommended Movie", "Year", "Rating"]] +
             [[i + 1, r["title"], r["year"], f"{r['vote_average']:.1f}"] for i, (_, r) in enumerate(ctx["demo_table"].iterrows())],
             widths=[0.7, 3.6, 0.9, 0.9])

    # ---------- chapter 7 ----------
    rp.h1("CHAPTER 7: STREAMLIT APPLICATION")
    rp.p("A web application named CineMatch was developed using Streamlit. It loads the trained artifacts and computes recommendations instantly for any selected movie.")
    rp.p("The application provides:")
    rp.bullets(["Recommend: choose a movie (or press “Surprise me”) and view the most similar movies, each with an explanation of why it was recommended (shared genres, director, cast and themes).",
                "My taste profile: choose 2 to 10 favourite movies to get recommendations that match the combination.",
                "Filters: genre, minimum rating, original language and release-year range.",
                "Engine selection: hybrid, embeddings, TF-IDF, Bag-of-Words or collaborative filtering, with adjustable hybrid weights.",
                "Explore the data: an interactive dashboard of the dataset (genres, ratings, decades, languages, top directors).",
                "Model comparison: the evaluation table and charts from Chapter 6.",
                "Lite mode: if the trained artifacts are missing, the app rebuilds a TF-IDF model directly from the CSV files so it still runs."])
    rp.p("The code is organised in three modules: features.py (data preparation), recommender.py (recommendation engine) and app.py (user interface). The same recommender module is used for evaluation in the notebook and in the deployed app.")
    rp.placeholder("[ Insert screenshot: Recommend tab with results ]")
    rp.placeholder("[ Insert screenshot: My taste profile tab ]")
    rp.placeholder("[ Insert screenshot: Explore the data tab ]")
    rp.placeholder("[ Insert screenshot: Model comparison tab ]")

    # ---------- chapter 8 ----------
    rp.h1("CHAPTER 8: RESULTS")
    rp.p("The project successfully developed and deployed a hybrid movie recommendation system.")
    rp.table([["Item", "Result"],
              ["Movies covered", f"{S['n_final']:,}"],
              ["Signals combined", "TF-IDF, sentence embeddings, collaborative filtering, rating/popularity prior"],
              ["Best method (Precision@10)", f"{best} ({_fmt(R.loc[best, 'Precision@10'])})"],
              ["Hybrid Precision@10 / Hit-rate@10", f"{_fmt(R.loc[hyb, 'Precision@10'])} / {_fmt(R.loc[hyb, 'HitRate@10'])}"],
              ["Original baseline Precision@10", _fmt(R.loc[base_name, 'Precision@10'])],
              ["Evaluation queries (test users)", f"{M['n_test_q']} ({M['n_test_users']} users)"],
              ["Deployment", "Streamlit web application"]], widths=[2.8, 3.6])

    # ---------- chapter 9 / 10 ----------
    rp.h1("CHAPTER 9: CONCLUSION")
    rp.p("This project demonstrated the complete pipeline of a recommendation system, from data preparation and exploratory analysis, through content-based, semantic and collaborative models, to quantitative evaluation and web deployment. "
         "Evaluating the methods on unseen users made it possible to compare the original Bag-of-Words approach with the improved models objectively, and the hybrid design shows how different sources of information can be combined. "
         "The project strengthened practical knowledge of natural language processing, matrix factorisation, evaluation methodology and software deployment.")
    rp.p("Limitations: the catalog is limited to the TMDB 5000 dataset; collaborative filtering is available only for movies with enough MovieLens ratings; and the offline evaluation treats unrated movies as not relevant, which underestimates real-world quality.")
    rp.h1("CHAPTER 10: FUTURE SCOPE")
    rp.bullets(["Use a larger movie catalog and the full MovieLens dataset.",
                "Add user accounts so that recommendations learn from each person's own ratings and watch history.",
                "Display posters and trailers through an API such as TMDB.",
                "Try neural collaborative filtering or session-based models.",
                "Run online A/B tests with real users to measure click-through and satisfaction.",
                "Add diversity and novelty measures so that recommendations are not all very similar to each other.",
                "Containerise the application and deploy it on a scalable cloud service."])
    rp.h1("REFERENCES")
    rp.bullets(["TMDB 5000 Movie Dataset (The Movie Database), movies and credits files.",
                "F. M. Harper and J. A. Konstan, “The MovieLens Datasets: History and Context”, ACM Transactions on Interactive Intelligent Systems, 2015.",
                "N. Reimers and I. Gurevych, “Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks”, EMNLP, 2019.",
                "F. Pedregosa et al., “Scikit-learn: Machine Learning in Python”, Journal of Machine Learning Research, 2011.",
                "Streamlit documentation, https://docs.streamlit.io"])
    d.save(path)
    return path

In [ ]:
# workflow diagram
from matplotlib.patches import FancyBboxPatch
fig, ax = plt.subplots(figsize=(8, 4.2)); ax.set_xlim(0, 10); ax.set_ylim(0, 5.2); ax.axis("off")
steps = ["Data\nCollection", "Preprocessing\n& EDA", "Feature\nEngineering", "Content Models\n(BoW, TF-IDF, Embed)",
         "Collaborative\nFiltering (SVD)", "Hybrid Ranking\n& Tuning", "Evaluation\n(Unseen users)", "Streamlit\nDeployment"]
for i, s_ in enumerate(steps):
    x, y = 0.2 + (i % 4) * 2.5, 3.6 if i < 4 else 1.2
    ax.add_patch(FancyBboxPatch((x, y), 2.0, 1.1, boxstyle="round,pad=0.02,rounding_size=0.12", fc=BLUE, ec=BLUE))
    ax.text(x + 1.0, y + 0.55, s_, ha="center", va="center", color="white", fontsize=9)
    if i % 4 != 3: ax.annotate("", (x + 2.45, y + 0.55), (x + 2.0, y + 0.55), arrowprops=dict(arrowstyle="->", color="#7F7F7F", lw=1.6))
ax.plot([8.7, 8.7, 1.2], [3.6, 2.95, 2.95], color="#7F7F7F", lw=1.6)
ax.annotate("", (1.2, 2.3), (1.2, 2.95), arrowprops=dict(arrowstyle="->", color="#7F7F7F", lw=1.6))
figs["workflow"] = f"{FIG_DIR}/workflow.png"; plt.savefig(figs["workflow"], dpi=170, bbox_inches="tight"); plt.show()

from make_report import build_report
top5 = final.recommend(q, n=5)
ctx = dict(
    stats=stats, results=results, best_w=best_w, figures=figs, emb_dim=int(E.shape[1]), svd_k=SVD_K,
    ml=dict(n_ratings=int(len(ratings)), n_users=int(ratings.userId.nunique()), n_movies_overlap=int(ratings.idx.nunique()),
            n_cf_movies=int(has_all.sum()), n_train_users=len(train_u), n_val_users=len(val_u), n_test_users=len(test_u),
            n_val_q=len(val_q), n_test_q=len(test_q)),
    demo_title=title, demo_table=top5)
print("report saved:", build_report(ctx, "Movie_Recommendation_Report.docx"))

## 11. Deploy
Upload these to your GitHub repository (repo root) and deploy on Streamlit Cloud with main file `app.py`:

```
app.py   features.py   recommender.py   requirements.txt   README.md
artifacts/                 <- created by this notebook
datasets/                  <- the two TMDB csv files (the app uses them if artifacts/ is missing)
```
Then open `Movie_Recommendation_Report.docx`, add your cover page and app screenshots, and update the index (right-click the index > *Update Field*).